# Predicting Annual Medical Insurance Premiums (Simplified)

**Goal:** build an explainable linear regression model that predicts `annual_premium`
from age, sex, BMI, children, smoker and region.

**Data:** public Medical Cost Personal Datasets (1,338 rows, no missing values).

**Steps:** 1 load -> 2 first look -> 3 explore -> 4 outliers -> 5 statistical tests ->
6 encode -> 7 split -> 8 check assumptions -> 9 build model -> 10 score -> 11 compare models

## 1. Setup and load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats                                   # t-test and ANOVA
import statsmodels.api as sm                              # OLS table with p-values
from statsmodels.stats.outliers_influence import variance_inflation_factor   # VIF
from statsmodels.stats.diagnostic import het_breuschpagan                    # constant-variance test

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

TARGET = "annual_premium"      # what we predict
RANDOM_STATE = 42              # makes the random split repeatable
sns.set_style("whitegrid")

In [ ]:
# In Colab, first run:  from google.colab import files; files.upload()
df = pd.read_csv("insurance_premium.csv")
if "charges" in df.columns:                      # original column name in the public dataset
    df = df.rename(columns={"charges": TARGET})
df.head()

## 2. First look at the data
`info()` shows column types, `describe()` shows averages and spread, and the missing-value
check tells us whether anything needs filling.

In [ ]:
df.info()
df.describe().T

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

# Fill gaps only if there are any: median for numbers, most common value for text
for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

numeric_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Numeric features    :", numeric_cols)
print("Categorical features:", categorical_cols)

## 3. Explore the data (EDA)
* Is the premium skewed? (a few very expensive policies)
* Which groups pay more?
* How do age and BMI relate to the premium?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], kde=True, ax=ax[0])
ax[0].set_title("Distribution of annual_premium")
sns.boxplot(x=df[TARGET], ax=ax[1])
ax[1].set_title("Box plot of annual_premium")
plt.tight_layout()
plt.show()
print(f"Skewness: {df[TARGET].skew():.2f}   (above 1 = long right tail)")

In [ ]:
fig, axes = plt.subplots(1, len(categorical_cols), figsize=(4.5 * len(categorical_cols), 4))
for ax, col in zip(axes, categorical_cols):
    sns.boxplot(x=col, y=TARGET, data=df, ax=ax)
    ax.set_title(f"Premium by {col}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["age", "bmi"]):
    sns.scatterplot(x=col, y=TARGET, hue="smoker", data=df, alpha=0.5, ax=ax)
    ax.set_title(f"{col} vs premium")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation: strength of straight-line relationship between numeric columns (-1 to +1)
corr = df[numeric_cols + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

## 4. Outliers (IQR rule)
A value is an outlier if it is below Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR.
We **cap the features** at these limits but **leave the premium alone**: expensive
premiums are real and are exactly what the model must learn.

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

df_clean = df.copy()
for col in numeric_cols + [TARGET]:
    low, high = iqr_bounds(df[col])
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col:15s} outliers: {n_out}")
    if col != TARGET:                                   # cap features only
        df_clean[col] = df_clean[col].clip(lower=low, upper=high)

## 5. Do the categories matter? (hypothesis tests)
Rule: **p < 0.05** means the difference is unlikely to be chance.
* **Welch t-test** compares the average premium of two groups.
* **ANOVA** compares the average premium of three or more groups.

In [ ]:
smokers = df_clean.loc[df_clean["smoker"] == "yes", TARGET]
non_smokers = df_clean.loc[df_clean["smoker"] == "no", TARGET]
t_stat, p_val = stats.ttest_ind(smokers, non_smokers, equal_var=False)
print(f"Smoker vs non-smoker: mean {smokers.mean():,.0f} vs {non_smokers.mean():,.0f}, p = {p_val:.3e}")

groups = [g[TARGET] for _, g in df_clean.groupby("region")]
f_stat, p_val = stats.f_oneway(*groups)
print(f"Region ANOVA: p = {p_val:.3f}  ->", "significant" if p_val < 0.05 else "not significant")

## 6. Turn text into numbers (encoding)
* Two-value columns -> 0 / 1.
* Region (4 values) -> one 0/1 column per region. We drop one (`drop_first=True`) so the
  columns are not perfectly predictable from each other (the dummy variable trap).

In [ ]:
df_enc = df_clean.copy()
df_enc["sex"] = df_enc["sex"].map({"female": 0, "male": 1})
df_enc["smoker"] = df_enc["smoker"].map({"no": 0, "yes": 1})
df_enc = pd.get_dummies(df_enc, columns=["region"], drop_first=True, dtype=int)
df_enc.head()

## 7. Train / test split
80% to train the model, 20% kept aside to test it on unseen data.

In [ ]:
X = df_enc.drop(columns=[TARGET])
y = df_enc[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)
print("Train rows:", len(X_train), "  Test rows:", len(X_test))

## 8. Check the regression assumptions
Linear regression works best when:
1. **Features are not too similar to each other** - VIF below 5 is good (above 10 is a problem).
2. **Error spread is constant** - Breusch-Pagan p above 0.05 is good.
3. **Errors look random** - the residual plot should be a shapeless cloud around zero.
4. **Errors are roughly normal** - Q-Q plot points should follow the diagonal.

A *residual* is actual minus predicted.

In [ ]:
X_train_sm = sm.add_constant(X_train)          # adds the intercept column statsmodels needs
ols = sm.OLS(y_train, X_train_sm).fit()
print(ols.summary())                            # coefficients, p-values, R-squared

In [ ]:
# 1. Multicollinearity (skip the constant in column 0)
vif = pd.Series([variance_inflation_factor(X_train_sm.values, i) for i in range(1, X_train_sm.shape[1])],
                index=X_train.columns, name="VIF").round(2)
print(vif.sort_values(ascending=False))

# 2. Constant variance
bp_p = het_breuschpagan(ols.resid, X_train_sm)[1]
print(f"\nBreusch-Pagan p-value = {bp_p:.4g} ->",
      "constant variance" if bp_p > 0.05 else "variance changes (common with cost data)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(ols.fittedvalues, ols.resid, alpha=0.4)
ax[0].axhline(0, color="red", linestyle="--")
ax[0].set(xlabel="Fitted values", ylabel="Residuals", title="Residuals vs fitted (look for curves or funnels)")
sm.qqplot(ols.resid, line="45", fit=True, ax=ax[1])
ax[1].set_title("Q-Q plot of residuals")
plt.tight_layout()
plt.show()

## 9. Build the linear model and read the coefficients
* `usd_per_unit`: the extra premium for one more unit of that feature (one more year of age, one more BMI point, smoker vs not...).
* `usd_per_1_std`: the effect of a typical-sized change, so features can be compared fairly.

In [ ]:
lin_reg = LinearRegression().fit(X_train, y_train)

coef_df = pd.DataFrame({
    "feature": X.columns,
    "usd_per_unit": lin_reg.coef_,
    "usd_per_1_std": lin_reg.coef_ * X_train.std().values,
}).sort_values("usd_per_1_std", key=abs, ascending=False)

print(f"Intercept: {lin_reg.intercept_:,.0f}")
coef_df.round(1)

## 10. How good is it?
* **R-squared:** share of the variation in premiums the model explains.
* **RMSE / MAE:** typical error in dollars (RMSE punishes big misses more).
* Train R-squared much higher than test R-squared = overfitting.
* **Cross-validation** repeats the train/test idea 5 times for a steadier score.

In [ ]:
train_pred = lin_reg.predict(X_train)
test_pred = lin_reg.predict(X_test)

print(f"Train R2 = {r2_score(y_train, train_pred):.3f}")
print(f"Test  R2 = {r2_score(y_test, test_pred):.3f}")
print(f"Test RMSE = {np.sqrt(mean_squared_error(y_test, test_pred)):,.0f} USD")
print(f"Test MAE  = {mean_absolute_error(y_test, test_pred):,.0f} USD")

cv = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()),
                     X_train, y_train, cv=5, scoring="r2")
print(f"5-fold CV R2 = {cv.mean():.3f} (+/- {cv.std():.3f})")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, test_pred, alpha=0.4)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual premium")
plt.ylabel("Predicted premium")
plt.title("Actual vs predicted (closer to the red line is better)")
plt.show()

## 11. Compare with Ridge and Lasso (regularisation)
These add a penalty on large coefficients to reduce overfitting.
* **Ridge** shrinks coefficients.
* **Lasso** can shrink some all the way to zero (feature selection).

They need scaled features, so each model is wrapped with a `StandardScaler` in a pipeline.
Adjusted R-squared = 1 - (1 - R2) x (n - 1) / (n - p - 1) penalises useless features.

In [ ]:
models = {
    "Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge":  make_pipeline(StandardScaler(), RidgeCV(alphas=[0.1, 1, 10, 100])),
    "Lasso":  make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=20000, random_state=RANDOM_STATE)),
}

def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    rows.append({
        "model": name,
        "test_R2": r2,
        "adjusted_R2": adjusted_r2(r2, len(y_test), X_test.shape[1]),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE": mean_absolute_error(y_test, pred),
    })

results = pd.DataFrame(rows).round(3)
results

In [ ]:
lasso_coefs = pd.Series(models["Lasso"][-1].coef_, index=X.columns)
zeroed = lasso_coefs[lasso_coefs.abs() < 1e-9].index.tolist()
best = results.sort_values("test_R2", ascending=False).iloc[0]

print("KEY RESULTS")
print("- Strongest drivers:", ", ".join(coef_df["feature"].head(3)))
print(f"- Best model on test data: {best['model']} (R2 = {best['test_R2']:.3f}, RMSE = {best['RMSE']:,.0f} USD)")
print("- Features removed by Lasso:", zeroed if zeroed else "none")
print("\nNext steps: log-transform the premium, add a smoker x BMI interaction, try a tree-based model.")